#### Visualized embeddings

In [ ]:
## Imports and code: 

%load_ext autoreload
%autoreload 2


import torch
import mlflow
import torch
import torch.nn as nn

from ra_utils.data.dataloader_CR_patches import (
    process_several_score_groups,
    dataset_and_loader_several,
    check_duplicates_in_dataloader
)

from ra_utils.training.scores_SHS.scores_SHS_training_lib_AE_v1 import (
    evaluate_and_log_testset_results_AE_v3,
    train_loop_AE_v3
)
from ra_utils.training.scores_SHS.scores_SHS_training_lib_AE_v4 import (
    evaluate_and_log_testset_results_AE_v4,
    train_loop_AE_v4
)

from ra_utils.networks.loss_function import get_score_loss_function, get_triplet_loss_fn
import torchvision.transforms.v2 as v2
from ra_utils.training.scores_SHS.model_builders import build_models_AE_v1_and2
import ra_utils.utils.utils_torch
from ra_utils.utils.verbosity_enums import *
import ra_utils.utils.utils

import ra_utils.utils.utils_torch
from pprint import pprint
import ra_utils.utils.config_parser

from ra_utils.utils.utils import datestr_to_years_since_2000


import numpy as np


from ra_utils.training.scores_SHS.run_training_main_lib import (
    check_config_consistency_and_partially_make_consistent,
    maybe_partially_init_model_from_state_dict,
)

import ra_utils.training.scores_SHS.run_training_main_lib

import ra_utils.loss.loss_fn_dict


import ra_utils.loss.loss_RnC_with_ranks_and_ids
import ra_utils.loss.loss_RnC

import ra_utils.loss.loss_RnCMono

import ra_utils.loss.online_mining_triplet_loss_MDP


import ra_utils.visualization.interactive.ra_model

from pathlib import Path

import ra_utils.visualization
import  ra_utils.visualization.trajectories

import pandas as pd
from collections import Counter



In [ ]:
config, config_name, config_originals = ra_utils.utils.config_parser.load_config(
    default_config="/msc/home/cwatze93/data/mlflow/mlflow_RA/467349521107954849/656df86cf7ff4b82918eaa571638f67b/artifacts/config_for_visualization_v2.yml",
    debugging_in_jupyter_nb=True, 
    silencium=True, 
    return_config_name=True, 
    return_originals=True
    )

# print("PIPII: ", config["data"]["classifier_head_infos"]["PIPII"])

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
ps = config.get("plot_settings", {}) or {}

loader_type  = ps.get("loader_type", "val").lower()  # "val" or "train"
split_key    = ps.get("val_key", "ALL_wo_wrist")     # used for train/val
png_root     = ps.get("png_root", "/home/cwatzenboeck/data/fo_png_cache/val")

cache_dir_save   = ps.get("cache_dir_save", None)
cache_dir_reload = ps.get("cache_dir_reload", None)
SAVE             = bool(ps.get("SAVE_EMBEDDINGS", False))
RELOAD           = bool(ps.get("RELOAD_EMBEDDINGS", False))

methods = tuple(ps.get("dimension_reduction_techniques", ["umap", "pca"]))

# -------------------- 1) Either reload pack or compute it
pack = None
used_cache_dir = None

if RELOAD:
    if not cache_dir_reload:
        raise ValueError("RELOAD_EMBEDDINGS=True but 'plot_settings.cache_dir_reload' is not set")
    used_cache_dir = Path(str(cache_dir_reload))
    print(f"[main] Reloading pack from: {used_cache_dir}")
    pack = ra_utils.visualization.interactive.ra_model.load_pack(str(used_cache_dir))

In [ ]:
# Get value counts of patient_scoretype_key to find the 5 longest trajectories


# Count occurrences of each patient_scoretype_key
patient_counts = Counter(pack["patient_scoretype_key"])
print("Top 10 longest trajectories:")
for i, (patient_key, count) in enumerate(patient_counts.most_common(10)):
    print(f"{i+1}. {patient_key}: {count} time points")

# Get the 5 longest trajectories
top_5_patients = [patient for patient, count in patient_counts.most_common(5)]
print(f"\nTop 5 longest trajectories: {top_5_patients}")
patient_counts.most_common(40)

In [ ]:
# Filter data for the top 5 longest trajectories
mask_top5 = np.isin(pack["patient_scoretype_key"], top_5_patients)

# Get filtered data
filtered_pack = {key: pack[key][mask_top5] for key in pack.keys()}

print(f"Filtered data shape: {len(filtered_pack['patient_scoretype_key'])} samples")
print(f"Unique patients in filtered data: {len(np.unique(filtered_pack['patient_scoretype_key']))}")

# Check if we have t_rel in the pack, if not compute it
if 't_rel' not in filtered_pack:
    print("Computing relative time...")
    from ra_utils.visualization.interactive.ra_model import compute_relative_time
    filtered_pack['t_rel'] = compute_relative_time(
        filtered_pack['years_since_2000'], 
        filtered_pack['patient_scoretype_key']
    )
    print("Relative time computed and added to pack")
else:
    print("t_rel already present in pack")


In [ ]:

unique, counts = np.unique(np.array(pack["patient_scoretype_key"]), return_counts=True)
# counts


# unique





In [ ]:
# Demonstrate the new separated workflow
print("=== New Workflow: Separate Computation and Plotting ===")

# Step 1: Compute 2D projections (this can take a while)
print("\n1. Computing t-SNE projection (this may take a while)...")
coords_tsne, reducer_tsne = ra_utils.visualization.trajectories.compute_2d_projection(pack, reduction_method="tsne", seed=43)

print("\n2. Computing PCA projection (faster)...")
coords_pca, reducer_pca = ra_utils.visualization.trajectories.compute_2d_projection(pack, reduction_method="pca", seed=43)

print("\n3. Computing UMAP projection...")
coords_umap, reducer_umap = ra_utils.visualization.trajectories.compute_2d_projection(pack, reduction_method="umap", seed=43)




In [ ]:
# Demonstrate background coloring with heatmaps
print("=== Background Coloring with Heatmaps ===")

# Show available keys in pack
print(f"Available keys in pack: {list(pack.keys())}")

# Example 1: Default gray background
print("\n1. Default gray background")
fig1, ax1 = ra_utils.visualization.trajectories.plot_trajectories_from_coords(
    coords_pca, pack, 
    trajectories_to_connect=["H_R_SMD3_181_MCPIIIED"], 
    reduction_method="pca", 
    traj_colors_columns="score_gt",
    traj_colors_colorbar=True
)


In [ ]:
# Example 3: Show trajectory colorbar for score_gt
print("\n3. Trajectory colorbar for score_gt")
fig3, ax3 = ra_utils.visualization.trajectories.plot_trajectories_from_coords(
    coords_pca, pack, 
    trajectories_to_connect=["H_R_SMD3_181_MCPIIIED"], 
    reduction_method="pca",
    #
    #traj_colors_columns="score_gt",
    traj_colors_columns=None,
    #traj_colors_colorbar=True,
    #
    background_color_key="score_gt",
    background_colormap="plasma"    
)

In [ ]:
# Example 3: Show trajectory colorbar for score_gt
print("\n3. Trajectory colorbar for score_gt")
fig3, ax3 = ra_utils.visualization.trajectories.plot_trajectories_from_coords(
    coords_pca, pack, 
    trajectories_to_connect=["H_R_SMD3_181_MCPIIIED"], 
    reduction_method="pca",
    #
    #traj_colors_columns="score_gt",
    traj_colors_columns=None,
    #traj_colors_colorbar=True,
    #
    background_color_key="t_rel",
    background_colormap="plasma"    
)